In [ ]:
import os
import gc
import pickle
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests

import jax
import jax.numpy as jnp
from jax import random
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
import arviz as az

warnings.filterwarnings('ignore')

In [ ]:
def balm_continuous_model(Y, N, P, L, M, K, df=3.5):
    """
    Pure continuous latent space model using Student-t likelihood.
    No Hurdle/Bernoulli component to maintain smooth Riemannian geometry.
    """
    alpha = 1.0

    tau = numpyro.sample("tau", dist.HalfNormal(2.0))
    sigma = numpyro.sample("sigma", dist.HalfNormal(2.0)) + 0.01

    q_templates = jnp.zeros((M, P))
    for m in range(M):
        U = numpyro.sample(f"U_{m}", dist.Normal(0, 1).expand([N, K]))
        gamma = numpyro.sample(f"gamma_{m}", dist.Normal(0, 1).expand([K]))

        Q_matrix, R_matrix = jnp.linalg.qr(U)
        Q_m = tau * jnp.dot(Q_matrix * gamma, Q_matrix.T)

        triu_idx = jnp.triu_indices(N, k=1)
        q_m_vec = Q_m[triu_idx]
        q_templates = q_templates.at[m, :].set(q_m_vec)

    with numpyro.plate("subjects", L):
        W = numpyro.sample("W", dist.Dirichlet(jnp.ones(M) * alpha))

    mu = jnp.dot(W, q_templates)

    with numpyro.plate("edges", P):
        with numpyro.plate("obs_subjects", L):
            numpyro.sample("Y_obs", dist.StudentT(df, mu, sigma), obs=Y)

In [ ]:
def run_continuous_model_search(npy_file_path, K_rank=2, max_M=6, save_dir="./continuous_balm_outputs"):
    os.makedirs(save_dir, exist_ok=True)

    print(f"Loading data from {npy_file_path}...")
    Y_train = np.load(npy_file_path)
    L, P = Y_train.shape
    N = int((1 + np.sqrt(1 + 8 * P)) / 2)
    print(f"Dataset Info -> Subjects (L): {L}, Nodes (N): {N}, Edges (P): {P}")

    Y_train_jax = jnp.array(Y_train, dtype=jnp.float32)

    m_candidates = list(range(2, max_M + 1))
    results_summary = {}

    for M in m_candidates:
        print(f"\n{'='*50}")
        print(f"Starting MCMC for M = {M} (K = {K_rank})")
        print(f"{'='*50}")

        rng_key = random.PRNGKey(M)

        kernel = NUTS(balm_continuous_model, target_accept_prob=0.85)
        mcmc = MCMC(
            kernel,
            num_warmup=1000,
            num_samples=3000,
            num_chains=1,
            progress_bar=True
        )

        mcmc.run(rng_key, Y=Y_train_jax, N=N, P=P, L=L, M=M, K=K_rank)
        posterior_samples = mcmc.get_samples()

        print(f"Calculating WAIC for M = {M}...")
        inference_data = az.from_numpyro(mcmc)
        waic_result = az.waic(inference_data, var_name="Y_obs", pointwise=False)

        waic_score = getattr(waic_result, "elpd_waic", getattr(waic_result, "waic", None))
        print(f"ELPD WAIC (M={M}): {waic_score:,.2f} (Higher is better)")

        results_summary[M] = {
            'elpd_waic': float(waic_score),
            'p_waic': float(getattr(waic_result, "p_waic", 0.0))
        }

        base_name = os.path.basename(npy_file_path).replace('.npy', '')
        save_path = os.path.join(save_dir, f"{base_name}_posterior_M{M}.pkl")
        with open(save_path, "wb") as f:
            pickle.dump(posterior_samples, f)
        print(f"Posterior saved to {save_path}")

        del mcmc
        del posterior_samples
        del inference_data
        gc.collect()
        jax.clear_caches()

    summary_path = os.path.join(save_dir, f"{base_name}_waic_summary.pkl")
    with open(summary_path, "wb") as f:
        pickle.dump(results_summary, f)

    print("\n" + "="*50)
    print(f"Model search complete for {base_name}. Summary saved.")

    best_m = max(results_summary, key=lambda m: results_summary[m]['elpd_waic'])
    print(f"RECOMMENDED MODEL: M = {best_m} (Highest ELPD WAIC: {results_summary[best_m]['elpd_waic']:,.2f})")
    print("="*50)

In [ ]:
def load_weights_and_merge(posterior_file, ids_file, csv_file, target_m):
    print(f"Loading posterior from {posterior_file}...")
    with open(posterior_file, "rb") as f:
        samples = pickle.load(f)

    W_mean = np.mean(samples['W'], axis=0)
    subject_ids = np.load(ids_file)

    df_W = pd.DataFrame(W_mean, columns=[f"Template_{m+1}" for m in range(target_m)])
    df_W.insert(0, "Subject", subject_ids.astype(str))

    print(f"Loading behavior data from {csv_file}...")
    df_beh = pd.read_csv(csv_file)
    df_beh['Subject'] = df_beh['Subject'].astype(str)

    df_merged = pd.merge(df_W, df_beh, on="Subject", how="inner")
    print(f"Merged Data Shape: {df_merged.shape[0]} subjects, {df_merged.shape[1]} columns.")

    return df_merged

def run_behavioral_trawl(df_merged, target_m, behavior_cols=None):
    templates = [f"Template_{m+1}" for m in range(target_m)]

    if behavior_cols is None:
        behavior_cols = df_merged.select_dtypes(include=[np.number]).columns.tolist()
        behavior_cols = [c for c in behavior_cols if not c.startswith("Template_")]

    results = []
    print(f"Scanning {len(behavior_cols)} behavioral columns across {target_m} templates...")

    for template in templates:
        w_values = df_merged[template]
        q25, q75 = np.percentile(w_values, 25), np.percentile(w_values, 75)

        low_group = df_merged[df_merged[template] <= q25]
        high_group = df_merged[df_merged[template] >= q75]

        for behavior in behavior_cols:
            val_low = pd.to_numeric(low_group[behavior], errors='coerce').dropna()
            val_high = pd.to_numeric(high_group[behavior], errors='coerce').dropna()

            if len(val_low) > 30 and len(val_high) > 30:
                t_stat, p_val = stats.ttest_ind(val_high, val_low, equal_var=False)

                if not np.isnan(p_val):
                    mean_diff = val_high.mean() - val_low.mean()
                    results.append({
                        "Template": template,
                        "Behavior": behavior,
                        "High_Exp_Mean": val_high.mean(),
                        "Low_Exp_Mean": val_low.mean(),
                        "Mean_Difference": mean_diff,
                        "T_Stat": t_stat,
                        "Raw_P_Value": p_val
                    })

    df_results = pd.DataFrame(results)

    if not df_results.empty:
        reject, pvals_corrected, _, _ = multipletests(df_results["Raw_P_Value"], alpha=0.05, method='fdr_bh')
        df_results["FDR_P_Value"] = pvals_corrected
        df_results["Significant (FDR < 0.05)"] = reject
        df_results = df_results.sort_values("Raw_P_Value")

    return df_results

In [ ]:
print(f"Detected JAX backend: {jax.default_backend()}")
print(f"Available devices: {jax.devices()}")

TARGET_DATASET_D15 = 'hcp_d15_continuous_edges.npy'
OUTPUT_DIRECTORY_D15 = './continuous_balm_outputs_d15'
run_continuous_model_search(npy_file_path=TARGET_DATASET_D15, K_rank=2, max_M=6, save_dir=OUTPUT_DIRECTORY_D15)

M_TARGET = 5
DATASET_PREFIX = "hcp_d15" 

POSTERIOR_FILE = f"./continuous_balm_outputs_{DATASET_PREFIX[-3:]}/{DATASET_PREFIX}_continuous_edges_posterior_M{M_TARGET}.pkl"
IDS_FILE = f"{DATASET_PREFIX}_continuous_edges_ids.npy"
CSV_FILE = "sample_info.csv"

TARGET_BEHAVIORS = [
    "PicVocab_Unadj", "PMAT24_A_CR", "ReadEng_Unadj", "PicSeq_Unadj", "CardSort_Unadj",
    "Flanker_Unadj", "VSPLOT_TC", "ListSort_Unadj", "CogFluidComp_Unadj", "CogCrystalComp_Unadj",
    "AngAffect_Unadj", "FearAffect_Unadj", "Sadness_Unadj", "LifeSatisf_Unadj",
    "NEOFAC_A", "NEOFAC_O", "NEOFAC_C", "NEOFAC_N", "NEOFAC_E",
    "PSQI_Score", "Endurance_Unadj", "GaitSpeed_Comp"
]

merged_data = load_weights_and_merge(POSTERIOR_FILE, IDS_FILE, CSV_FILE, M_TARGET)
scan_results = run_behavioral_trawl(merged_data, M_TARGET, behavior_cols=TARGET_BEHAVIORS)

print("\n=== Top 15 Behavioral Associations ===")
display_cols = ["Template", "Behavior", "Mean_Difference", "Raw_P_Value", "FDR_P_Value", "Significant (FDR < 0.05)"]
print(scan_results[display_cols].head(15).to_string(index=False))